## Using Neurosynth to decode the psychological processes that have been found to be related to the clusters
## Stable version
## Xiao Chen 261119
## chenxiaophd@gmail.com

In [ ]:
import os
import pandas as pd
from pprint import pprint

from neuromaps import transforms
from neuromaps.resampling import resample_images

import nibabel as nib
import numpy as np
import openpyxl
from nilearn.plotting import plot_roi
from nilearn.image import resample_to_img

from nimare.dataset import Dataset
from nimare.decode import discrete
from nimare.utils import get_resource_path
from nimare.extract import download_abstracts, fetch_neuroquery, fetch_neurosynth
from nimare.io import convert_neurosynth_to_dataset

import Bio

from pathlib import Path
import subprocess

In [ ]:
result_dir = "/xxxxxx/decode"
data_dir = "/xxxxxx/data"
out_dir = os.path.abspath("xxxxxxxx/data")
os.makedirs(out_dir, exist_ok=True)
output_dir = "/xxxxxxx/decode"
os.makedirs(output_dir, exist_ok=True)

## load the compressed dataset

In [ ]:
# dset = neurosynth_dset
dset = Dataset.load(os.path.join(out_dir, "neurosynth_dataset.pkl.gz"))
# dset.annotations.head(5)

## do decode

In [ ]:
roi_label = "positive_gt3p29_cluster3_n33_mask"
project_root = Path("/Users/Chenxiao/Documents/project_codes/rumination_casual_mapping")
surface_mask_path = project_root / f"analyses/stable_260725/thresholded_maps/{roi_label}.func.gii"
surface2vol_script = project_root / "scripts/surface2vol.sh"

conversion_env = os.environ.copy()
conversion_env.setdefault("FREESURFER_HOME", "/Applications/freesurfer")
conversion_env.setdefault("SUBJECTS_DIR", "/Applications/freesurfer/subjects")
conversion_env.setdefault("FSLDIR", "/Users/ChenXiao/Documents/MATLAB/fsl")

subprocess.run(
    [str(surface2vol_script), "--hemi", "rh", "--interp", "nearest", str(surface_mask_path)],
    check=True,
    cwd=project_root,
    env=conversion_env,
)

mni_mask_path = surface_mask_path.with_name(
    surface_mask_path.name.removesuffix(".func.gii") + "_MNI152.nii"
)
print(f"MNI mask: {mni_mask_path}")

In [ ]:
nii = nib.load(mni_mask_path)
data = nii.get_fdata()
binary_data = (data != 0).astype(np.uint8)
binary_nii = nib.Nifti1Image(binary_data, nii.affine, nii.header)
# nib.save(binary_nii, os.path.join(result_dir, "positive_gt2_cluster1_n309_MNI152_mask.nii.gz"))

ref_img = dset.masker.mask_img
src_mask_img = nib.Nifti1Image(binary_data.astype(np.int32), nii.affine, nii.header)

mask_img = resample_to_img(
    src_mask_img,
    ref_img,
    interpolation="nearest"
)
plot_roi(mask_img, draw_cross=False)

# Get studies with voxels in the mask
ids = dset.get_studies_by_mask(mask_img)


## Decode an ROI image using the Neurosynth chi-square method
## this method seems most optimal

In [ ]:
# Run the decoder
decoder = discrete.NeurosynthDecoder(correction=None)
decoder.fit(dset)
decoded_df = decoder.transform(ids=ids)
decoded_df.sort_values(by="probReverse", ascending=False).head(20)

In [ ]:
top50 = (
    decoded_df
    .sort_values(by="probReverse", ascending=False)
    .head(50)
)
out_df = pd.DataFrame({
    "Terms": top50.index.str.split("__").str[-1],         
    "corr": top50["probReverse"]
})
out_path = os.path.join(output_dir, f"neurosynth_decoder_top50_{roi_label}.xlsx")
out_df.to_excel(out_path, index=False)

## legacy code: download and compress the neurosynth dataset, legacy code, just need to do it once

In [ ]:
files = fetch_neurosynth(
    data_dir=out_dir,
    version="7",
    overwrite=False,
    source="abstract",
    vocab="terms",
)
# Note that the files are saved to a new folder within "out_dir" named "neurosynth".
pprint(files)
neurosynth_db = files[0]

neurosynth_dset = convert_neurosynth_to_dataset(
    coordinates_file=neurosynth_db["coordinates"],
    metadata_file=neurosynth_db["metadata"],
    annotations_files=neurosynth_db["features"],
)
neurosynth_dset.save(os.path.join(out_dir, "neurosynth_dataset.pkl.gz"))
print(neurosynth_dset)